# Elasticsearch RAG Experiment

This notebook uses Elasticsearch as the retrieval layer for the course FAQ RAG assistant.

## 1. Setup

Keep the Elasticsearch Docker container running before executing this notebook.

In [1]:
from pathlib import Path
import os
import sys

from dotenv import load_dotenv
from openai import OpenAI
import requests

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY was not loaded from .env")

openai_client = OpenAI()

## 2. Check Elasticsearch

In [2]:
response = requests.get("http://localhost:9200", timeout=10)
response.raise_for_status()
response.json()

{'name': 'ed7ac2af25b4',
 'cluster_name': 'docker-cluster',
 'cluster_uuid': 'TFHEkwXrTGauVdO4Lre2Jg',
 'version': {'number': '8.15.0',
  'build_flavor': 'default',
  'build_type': 'docker',
  'build_hash': '1a77947f34deddb41af25e6f0ddb8e830159c179',
  'build_date': '2024-08-05T10:05:34.233336849Z',
  'build_snapshot': False,
  'lucene_version': '9.11.1',
  'minimum_wire_compatibility_version': '7.17.0',
  'minimum_index_compatibility_version': '7.0.0'},
 'tagline': 'You Know, for Search'}

## 3. Load FAQ Documents

In [3]:
from ingest import load_faq_data

documents = load_faq_data()
len(documents), documents[0]

(1342,
 {'id': '9e508f2212',
  'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: When does the course start?',
  'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."})

## 4. Create and Populate Elasticsearch Index

In [4]:
from elastic_search import ElasticSearchIndex

es_index = ElasticSearchIndex(index_name="course-faq")
es_index.create_index(recreate=True)
es_index.index_documents(documents)

## 5. Test Retrieval

In [5]:
question = "I just discovered the course. Can I join now?"

results = es_index.search(
    query=question,
    filter_dict={"course": "llm-zoomcamp"},
    boost_dict={"question": 3.0, "section": 0.5},
    num_results=5,
)

results[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

## 6. Run RAG with Elasticsearch

In [6]:
from rag_helper import RAGBase

assistant = RAGBase(
    index=es_index,
    llm_client=openai_client,
)

answer = assistant.rag(question)
print(answer)

Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


## 7. Compare with Minsearch

In [7]:
from ingest import build_index

minsearch_index = build_index(documents)

minsearch_results = minsearch_index.search(
    query=question,
    filter_dict={"course": "llm-zoomcamp"},
    boost_dict={"question": 3.0, "section": 0.5},
    num_results=5,
)

minsearch_results[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [8]:
comparison = {
    "elasticsearch_top_question": results[0]["question"],
    "minsearch_top_question": minsearch_results[0]["question"],
}

comparison

{'elasticsearch_top_question': 'I just discovered the course. Can I still join?',
 'minsearch_top_question': 'I just discovered the course. Can I still join?'}